# QuickPay FinTech — Python Reconciliation Pipeline
**Parts 3 & 4: Reconciliation Workflow + JSON Normalization**

In [ ]:
import pandas as pd
import numpy as np
import json
import os

RAW  = '../01_data/raw'
PROC = '../01_data/processed'
os.makedirs(PROC, exist_ok=True)

## Part 3 — Reconciliation Workflow
### Step 1: Load ledger and gateway files

In [ ]:
ledger  = pd.read_csv(f'{RAW}/ledger.csv')
gateway = pd.read_csv(f'{RAW}/gateway.csv')

print('Ledger shape:', ledger.shape)
print('Gateway shape:', gateway.shape)
print()
print('--- Ledger ---')
display(ledger)
print('--- Gateway ---')
display(gateway)

### Step 2: Check for duplicates and nulls

In [ ]:
print('=== LEDGER ===')
print('Duplicate transaction_ids:', ledger['transaction_id'].duplicated().sum())
print('Null counts:')
print(ledger.isnull().sum())

print()
print('=== GATEWAY ===')
print('Duplicate transaction_ids:', gateway['transaction_id'].duplicated().sum())
print('Null counts:')
print(gateway.isnull().sum())

### Step 3: Standardize for comparison

In [ ]:
led = ledger.copy()
gw  = gateway.copy()

# Strip whitespace
led['transaction_id'] = led['transaction_id'].str.strip()
gw['transaction_id']  = gw['transaction_id'].str.strip()
led['status'] = led['status'].str.strip().str.lower()
gw['status']  = gw['status'].str.strip().str.lower()

led_ids = set(led['transaction_id'])
gw_ids  = set(gw['transaction_id'])

print('Ledger IDs:', sorted(led_ids))
print('Gateway IDs:', sorted(gw_ids))

### Step 4: Records missing in gateway (in ledger but not in gateway)

In [ ]:
missing_in_gateway = led[~led['transaction_id'].isin(gw_ids)].copy()
print(f'Missing in gateway: {len(missing_in_gateway)} record(s)')
display(missing_in_gateway)
missing_in_gateway.to_csv(f'{PROC}/missing_in_gateway.csv', index=False)
print('Saved: missing_in_gateway.csv')

### Step 5: Records missing in ledger (in gateway but not in ledger)

In [ ]:
missing_in_ledger = gw[~gw['transaction_id'].isin(led_ids)].copy()
print(f'Missing in ledger: {len(missing_in_ledger)} record(s)')
display(missing_in_ledger)
missing_in_ledger.to_csv(f'{PROC}/missing_in_ledger.csv', index=False)
print('Saved: missing_in_ledger.csv')

### Step 6: Identify amount mismatches

In [ ]:
common = led.merge(gw, on='transaction_id', suffixes=('_ledger', '_gateway'))

amount_mismatches = common[common['amount_usd_ledger'] != common['amount_usd_gateway']].copy()
amount_mismatches['amount_difference'] = (
    amount_mismatches['amount_usd_ledger'] - amount_mismatches['amount_usd_gateway']
).round(2)

print(f'Amount mismatches: {len(amount_mismatches)} record(s)')
display(amount_mismatches[['transaction_id','amount_usd_ledger','amount_usd_gateway','amount_difference']])
amount_mismatches.to_csv(f'{PROC}/amount_mismatches.csv', index=False)
print('Saved: amount_mismatches.csv')

### Step 7: Identify status mismatches

In [ ]:
status_mismatches = common[common['status_ledger'] != common['status_gateway']].copy()

print(f'Status mismatches: {len(status_mismatches)} record(s)')
display(status_mismatches[['transaction_id','status_ledger','status_gateway']])
status_mismatches.to_csv(f'{PROC}/status_mismatches.csv', index=False)
print('Saved: status_mismatches.csv')

### Step 8: Build final reconciliation report

In [ ]:
issue_rows = []

for _, row in missing_in_gateway.iterrows():
    issue_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'missing_in_gateway',
        'ledger_amount': row['amount_usd'],
        'gateway_amount': None,
        'ledger_status': row['status'],
        'gateway_status': None
    })

for _, row in missing_in_ledger.iterrows():
    issue_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'missing_in_ledger',
        'ledger_amount': None,
        'gateway_amount': row['amount_usd'],
        'ledger_status': None,
        'gateway_status': row['status']
    })

for _, row in amount_mismatches.iterrows():
    issue_rows.append({
        'transaction_id': row['transaction_id'],
        'issue_type': 'amount_mismatch',
        'ledger_amount': row['amount_usd_ledger'],
        'gateway_amount': row['amount_usd_gateway'],
        'ledger_status': row['status_ledger'],
        'gateway_status': row['status_gateway']
    })

amt_mm_ids = set(amount_mismatches['transaction_id'])
for _, row in status_mismatches.iterrows():
    if row['transaction_id'] not in amt_mm_ids:
        issue_rows.append({
            'transaction_id': row['transaction_id'],
            'issue_type': 'status_mismatch',
            'ledger_amount': row['amount_usd_ledger'],
            'gateway_amount': row['amount_usd_gateway'],
            'ledger_status': row['status_ledger'],
            'gateway_status': row['status_gateway']
        })

reconciliation_report = pd.DataFrame(issue_rows)
print(f'Total reconciliation issues: {len(reconciliation_report)}')
display(reconciliation_report)
reconciliation_report.to_csv(f'{PROC}/reconciliation_report.csv', index=False)
print('Saved: reconciliation_report.csv')

### Step 9: Generate summary metrics

In [ ]:
amount_at_risk = (
    missing_in_gateway['amount_usd'].sum() +
    missing_in_ledger['amount_usd'].sum() +
    amount_mismatches['amount_difference'].abs().sum()
)

metrics = {
    'total_ledger_rows': int(len(led)),
    'total_gateway_rows': int(len(gw)),
    'missing_in_gateway_count': int(len(missing_in_gateway)),
    'missing_in_ledger_count': int(len(missing_in_ledger)),
    'amount_mismatch_count': int(len(amount_mismatches)),
    'status_mismatch_count': int(len(status_mismatches)),
    'reconciliation_issue_count': int(len(reconciliation_report)),
    'ledger_total_amount': round(float(led['amount_usd'].sum()), 2),
    'gateway_total_amount': round(float(gw['amount_usd'].sum()), 2),
    'amount_at_risk': round(float(amount_at_risk), 2)
}

print(json.dumps(metrics, indent=2))
with open('../04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: summary_metrics.json')

---
## Part 4 — JSON Normalization

In [ ]:
with open(f'{RAW}/api_response_sample.json') as f:
    api_data = json.load(f)

print('Top-level keys:', list(api_data.keys()))
print('Number of batches:', len(api_data['batches']))

In [ ]:
rows = []
for batch in api_data['batches']:
    for settlement in batch['settlements']:
        rows.append({
            'generated_at':     api_data['generated_at'],
            'source':           api_data['source'],
            'batch_id':         batch['batch_id'],
            'merchant_id':      batch['merchant']['merchant_id'],
            'merchant_name':    batch['merchant']['merchant_name'],
            'merchant_region':  batch['merchant']['region'],
            'settlement_id':    settlement['settlement_id'],
            'amount_usd':       settlement['amount_usd'],
            'status':           settlement['status'],
            'processed_at':     settlement['processed_at'],
            'bank_name':        settlement['bank']['name'],
            'bank_country':     settlement['bank']['country']
        })

api_df = pd.DataFrame(rows)

# Convert datetime columns
api_df['generated_at'] = pd.to_datetime(api_df['generated_at'])
api_df['processed_at'] = pd.to_datetime(api_df['processed_at'])

print('Normalized shape:', api_df.shape)
display(api_df)

api_df.to_csv(f'{PROC}/api_normalized.csv', index=False)
print('Saved: api_normalized.csv')

### Settlement summary by merchant

In [ ]:
print('Settlement summary by merchant:')
display(api_df.groupby(['merchant_name', 'status'])['amount_usd'].sum().reset_index())

print('\nTotal settled amount (USD):', api_df[api_df['status']=='settled']['amount_usd'].sum())
print('Total pending amount (USD):', api_df[api_df['status']=='pending']['amount_usd'].sum())
print('Total failed amount (USD):', api_df[api_df['status']=='failed']['amount_usd'].sum())